# VF-1 Phase 3A — S1-mini LoRA smoke test (Colab)

**Status when shipped: NOT RUN.** This notebook is prepared, not executed.

The trainer refuses to train until `gates.json` records four approvals, each with who and when:
1. `data_verified`
2. `human_audit_approved` (a real person reviewed `HUMAN_AUDIT_SHEET`)
3. `permissions_cleared` (written permission to train on the Hark-generated data, or replacement data)
4. `user_gpu_approval`

The base checkpoint is never modified; only a LoRA adapter is saved. Full-corpus training is disabled in this phase.

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print('bf16 supported:', torch.cuda.is_available() and torch.cuda.is_bf16_supported())

In [ ]:
# Pinned versions the trainer was verified with on CPU (Phase 3A). Colab's torch is kept.
!pip -q install 'transformers==5.19.0' 'peft==0.21.2' 'accelerate==1.15.0' 'tokenizers==0.23.3' jinja2 jsonschema pytest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put vf1_phase2b.zip and vf1_phase3a.zip in MyDrive/APF_VF1/ first.
!mkdir -p /content/work && cd /content/work && unzip -oq /content/drive/MyDrive/APF_VF1/vf1_phase2b.zip && unzip -oq /content/drive/MyDrive/APF_VF1/vf1_phase3a.zip
%cd /content/work
!ls

In [ ]:
P2B='/content/work/vf1_phase2b/research/vf_phase2b'
T='/content/work/vf1_phase3a/research/vf_phase3a/S1_LORA_TRAINER.py'
# CPU-side checks again on Colab (hashes, renderer, collator, schedules, tiny random-token LoRA run)
!python $T verify --p2b $P2B --report /content/work/verify_colab.json

## Gates
Fill this only with real approvals. The trainer checks every field and stops otherwise.

In [ ]:
import json, datetime
gates = json.load(open('/content/work/vf1_phase3a/research/vf_phase3a/GATES_TEMPLATE.json'))
# Example (edit truthfully):
# gates['human_audit_approved'] = {'approved': True, 'by': 'Uzair Kayat', 'date': '2026-10-13', 'evidence': 'HUMAN_AUDIT_SHEET.csv, 0 critical errors'}
json.dump(gates, open('/content/work/gates.json','w'), indent=1)
print(json.dumps(gates, indent=1))

## Step 1 — tiny trainability check (16 common + 3 rehearsal rows, 8 epochs)
Expect training loss to fall clearly and the run to finish in a few minutes. Downloads the authentic `superwhisper/s1-mini` checkpoint (pinned revision).

In [ ]:
!python $T train --p2b $P2B --gates /content/work/gates.json --profile tiny --out /content/work/runs/tiny

## Step 2 — controlled smoke run (290 common rows across all 8 buckets + 51 rehearsal rows, 2 epochs, 15% rehearsal exposures)
On a T4 (fp16 autocast, fp32 adapter) expect roughly 40–80 optimizer steps. Use `--max-steps` to cap it.

In [ ]:
!python $T train --p2b $P2B --gates /content/work/gates.json --profile smoke --out /content/work/runs/smoke1

In [ ]:
import pandas as pd, json
m = pd.read_json('/content/work/runs/smoke1/metrics.jsonl', lines=True)
m[['step','loss_common','loss_rehearsal','val_loss','grad_norm','lr']].tail(20)

## Step 3 — baseline vs adapter on development examples
Validation split (60) + 20 held-back rehearsal rows. Never the frozen 200-case held-out set.

In [ ]:
!python $T compare --p2b $P2B --adapter /content/work/runs/smoke1/adapter --out /content/work/runs/smoke1/compare.jsonl

In [ ]:
import json
for l in open('/content/work/runs/smoke1/compare.jsonl'):
    r = json.loads(l)
    print('='*80); print(r['id'], r['bucket']); print('REQ:', {k:v for k,v in r['request'].items() if k!='transcript'}); print('IN :', r['request'].get('transcript','')[:300])
    print('REF:', r['reference'][:300]); print('BASE:', r['base'][:300], '| stopped', r['base_stopped']); print('ADPT:', r['adapter'][:300], '| stopped', r['adapter_stopped'])

In [ ]:
# Save the adapter (only) and its hashes to Drive
!mkdir -p /content/drive/MyDrive/APF_VF1/runs && cp -r /content/work/runs/smoke1 /content/drive/MyDrive/APF_VF1/runs/
!sha256sum /content/work/runs/smoke1/adapter/*